## Deep Research Agent — LangGraph

Agentic pipeline built with **LangGraph** and open-source models (Groq / Cerebras / OpenRouter).

**Pipeline:**
1. **Clarifier** — generates 3 scoping questions; user picks one and adds context
2. **Planner** — turns query + clarification into prioritised search terms
3. **Searcher** — runs all searches in parallel via DuckDuckGo (no API key needed)
4. **Sufficiency checker** — decides if evidence is enough; triggers up to 2 extra rounds
5. **Writer** — produces a structured Markdown report (1000+ words), streamed live
6. **Evaluator** — scores 0-10; approves or requests one revision
7. **Emailer** — sends the final report via SendGrid

**Key features:**
- SQLite checkpointer for persistent memory across sessions and kernel restarts
- Per-user isolation via unique session IDs — multiple users can research simultaneously
- Session resume: paste a previous session ID to pick up where you left off
- LLM-based input guardrail blocks harmful queries before any searching begins
- Heuristic output guardrail checks report quality (word count, placeholder detection)
- Writer output streamed live to the UI via `astream_events`
- Explicit model parameters (temperature, max_tokens) per agent role
- Provider switchable via `RESEARCH_PROVIDER` env var: `groq` | `cerebras` | `openrouter` | `openai`
- LangSmith tracing via `LANGCHAIN_API_KEY` (no inference tokens consumed)

In [ ]:
!uv pip install langgraph langchain-groq langchain-openai langchain-community ddgs sendgrid gradio python-dotenv pydantic

In [ ]:
from __future__ import annotations
import asyncio
import os
import sqlite3
import uuid
from typing import Any, Optional
from typing_extensions import TypedDict

import nest_asyncio
import sendgrid
from ddgs import DDGS
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from sendgrid.helpers.mail import Content, Email, Mail, To

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.sqlite import SqliteSaver

from IPython.display import Image, display
import gradio as gr

nest_asyncio.apply()

In [ ]:
load_dotenv(override=True)

PROVIDER = os.environ.get("RESEARCH_PROVIDER", "groq")

_PROVIDERS: dict[str, dict] = {
    "groq":       {"model": "llama-3.3-70b-versatile",           "base_url": "https://api.groq.com/openai/v1",    "api_key_env": "GROQ_API_KEY"},
    "cerebras":   {"model": "qwen-3-235b-a22b-instruct-2507",    "base_url": "https://api.cerebras.ai/v1",       "api_key_env": "CEREBRAS_API_KEY"},
    "openrouter": {"model": "meta-llama/llama-3.3-70b-instruct", "base_url": "https://openrouter.ai/api/v1",    "api_key_env": "OPENROUTER_API_KEY"},
    "openai":     {"model": "gpt-4o-mini",                       "base_url": "https://api.openai.com/v1",        "api_key_env": "OPENAI_API_KEY"},
}

_cfg = _PROVIDERS.get(PROVIDER, _PROVIDERS["groq"])

# PR checklist: explicit model parameters per agent role
# temperature controls creativity; max_tokens caps output length; top_p controls diversity
_MODEL_PARAMS = {
    "default":   {"temperature": 0.3, "max_tokens": 4096, "top_p": 0.95},
    "creative":  {"temperature": 0.6, "max_tokens": 4096, "top_p": 0.95},
    "structured":{"temperature": 0.1, "max_tokens": 2048, "top_p": 0.90},
    "safety":    {"temperature": 0.0, "max_tokens": 256,  "top_p": 1.00},
}


def _build_llm(role: str = "default") -> ChatOpenAI:
    """Return a ChatOpenAI-compatible LLM for the configured provider and role."""
    params = _MODEL_PARAMS[role]
    return ChatOpenAI(
        model=_cfg["model"],
        base_url=_cfg["base_url"],
        api_key=os.environ.get(_cfg["api_key_env"], ""),
        temperature=params["temperature"],
        max_tokens=params["max_tokens"],
        # streaming=True enables on_chat_model_stream events for live output
        streaming=True,
    )


# LangSmith tracing — uses LANGCHAIN_API_KEY, consumes no inference tokens
if os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "deep-research-langgraph"
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"

print(f"Provider: {PROVIDER} | Model: {_cfg['model']}")

In [ ]:
# Pydantic schemas for structured LLM outputs

class SafetyCheck(BaseModel):
    is_safe: bool = Field(description="True if the query is safe and appropriate to research.")
    reason:  str  = Field(description="Brief explanation of the safety decision.")


class ClarifyingQuestions(BaseModel):
    questions:       list[str] = Field(description="Three concise clarifying questions to focus the research scope.")
    context_summary: str       = Field(description="One-sentence summary of what is already understood from the query.")


class WebSearchPlan(BaseModel):
    queries: list[str] = Field(description="Five to eight prioritised web search terms, most important first.")


class ResearchSufficiency(BaseModel):
    is_sufficient:     bool      = Field(description="True if current evidence is enough for a thorough report.")
    additional_queries: list[str] = Field(default_factory=list, description="New search terms to fill gaps.")
    reasoning:         str       = Field(description="Brief explanation of the decision.")


class ReportEvaluation(BaseModel):
    score:       float     = Field(ge=0, le=10, description="Quality score 0-10.")
    is_approved: bool      = Field(description="True if score >= 7 and report meets quality standards.")
    feedback:    str       = Field(description="Evaluation feedback.")
    suggestions: list[str] = Field(description="Actionable improvement suggestions if not approved.")


# Shared state — the whiteboard every LangGraph node reads from and writes to
class ResearchState(TypedDict):
    query:                str         # original user query
    clarifying_questions: list[str]   # 3 questions from clarifier
    context_summary:      str         # one-line summary from clarifier
    user_clarification:   str         # user's selected/edited clarification
    search_plan:          list[str]   # search terms from planner
    search_results:       str         # accumulated search result text
    search_retries:       int         # extra search rounds triggered by sufficiency agent
    is_sufficient:        bool        # sufficiency verdict (read by router)
    report:               str         # markdown report from writer
    report_score:         float       # evaluator score
    report_feedback:      str         # evaluator feedback (fed back to writer on retry)
    report_retries:       int         # revision rounds triggered by evaluator
    report_approved:      bool        # evaluator verdict (read by router)
    email_status:         str         # sendgrid delivery status

In [ ]:
# Tools

@tool
def web_search(query: str) -> str:
    """Search the web using DuckDuckGo. Returns top 5 results with title, snippet and URL.
    No API key required — works with any LLM provider."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return f"No results found for: {query}"
        return "\n\n---\n\n".join(
            f"{r.get('title', '')}\n{r.get('body', '')}\nSource: {r.get('href', '')}"
            for r in results
        )
    except Exception as exc:
        return f"Search error for '{query}': {exc}"


def send_report_email(subject: str, body: str) -> str:
    """Send the research report as an email via SendGrid.
    All credentials read from environment variables — nothing hardcoded."""
    api_key   = os.environ.get("SENDGRID_API_KEY")
    from_addr = os.environ.get("SENDGRID_FROM_EMAIL", "sender@example.com")
    to_addr   = os.environ.get("SENDGRID_TO_EMAIL",   "recipient@example.com")
    if not api_key:
        return "error: SENDGRID_API_KEY not set"
    try:
        sg   = sendgrid.SendGridAPIClient(api_key=api_key)
        mail = Mail(
            Email(from_addr), To(to_addr), subject,
            Content("text/html", body.replace("\n", "<br>"))
        )
        res = sg.client.mail.send.post(request_body=mail.get())
        return f"sent (status {res.status_code})"
    except Exception as exc:
        return f"error: {exc}"

In [ ]:
# LLM instances — separate role-based configurations (PR checklist: model parameters)
default_llm    = _build_llm("default")    # planners, sufficiency, clarifier
creative_llm   = _build_llm("creative")   # writer — higher temperature for prose
structured_llm = _build_llm("structured") # evaluator — low temp for consistent scoring
safety_llm_raw = _build_llm("safety")     # safety classifier — deterministic (temp=0)

# with_structured_output uses tool-calling (not json_schema) — works on Groq/Cerebras
clarifier_llm   = default_llm.with_structured_output(ClarifyingQuestions)
planner_llm     = default_llm.with_structured_output(WebSearchPlan)
sufficiency_llm = default_llm.with_structured_output(ResearchSufficiency)
evaluator_llm   = structured_llm.with_structured_output(ReportEvaluation)
safety_llm      = safety_llm_raw.with_structured_output(SafetyCheck)

In [ ]:
# ── Guardrails ─────────────────────────────────────────────────────────────────

def check_query_safety(query: str) -> SafetyCheck:
    """
    Input guardrail: LLM-based safety classifier.
    Runs before the clarifier. Blocks queries that request harmful, illegal,
    or harassing content. Fails open (allows request) if the classifier itself errors.
    """
    try:
        return safety_llm.invoke([
            SystemMessage(
                "Classify whether this research query is safe. "
                "Flag UNSAFE for: instructions to cause harm, illegal activity, "
                "synthesis of dangerous substances, or harassment of private individuals. "
                "Legitimate research on sensitive topics (history, medicine, security) is SAFE."
            ),
            HumanMessage(query),
        ])
    except Exception:
        # Fail open — if classifier errors, do not block the user
        return SafetyCheck(is_safe=True, reason="classifier unavailable")


def check_report_quality(report: str) -> tuple[bool, str]:
    """
    Output guardrail: heuristic quality check on the generated report.
    Checks minimum word count and absence of unfilled placeholder text.
    Returns (passed, message).
    """
    if not report.strip():
        return False, "Report is empty"
    wc = len(report.split())
    if wc < 200:
        return False, f"Report too short ({wc} words, minimum 200)"
    for placeholder in ["[PLACEHOLDER]", "[TODO]", "TODO:", "FIXME", "[INSERT"]:
        if placeholder in report:
            return False, f"Report contains placeholder text: '{placeholder}'"
    return True, f"Quality check passed ({wc} words)"

In [ ]:
MAX_SEARCH_RETRIES = 2
MAX_REPORT_RETRIES = 1

# Node names used by the streaming router to filter events
_PIPELINE_NODES = {"planner", "searcher", "sufficiency", "writer", "evaluator", "emailer"}


# ── Node functions ─────────────────────────────────────────────────────────────

def clarifier_node(state: ResearchState) -> dict:
    """Generate 3 scoping questions and a context summary for the query."""
    result = clarifier_llm.invoke([
        SystemMessage("Generate exactly 3 clarifying questions to focus a research query, and a one-sentence context summary."),
        HumanMessage(state["query"]),
    ])
    return {"clarifying_questions": result.questions[:3], "context_summary": result.context_summary}


def planner_node(state: ResearchState) -> dict:
    """Turn the query + user clarification into a prioritised list of search terms."""
    extra = ""
    if state.get("search_retries", 0) > 0:
        extra = f"\nPrevious searches were insufficient. Expand coverage of: {state.get('search_plan', [])}"
    result = planner_llm.invoke([
        SystemMessage("You are a research planner. Produce 5-8 prioritised web search terms, most important first."),
        HumanMessage(
            f"Research query: {state['query']}\n"
            f"User clarification: {state.get('user_clarification', '')}\n{extra}"
        ),
    ])
    return {"search_plan": result.queries}


async def searcher_node(state: ResearchState) -> dict:
    """Run all planned searches in parallel using DuckDuckGo."""
    async def _one(q: str) -> str:
        try:
            result = await asyncio.to_thread(web_search.invoke, q)
            return f"### {q}\n{result}\n"
        except Exception as exc:
            return f"### {q}\nFailed: {exc}\n"

    parts       = await asyncio.gather(*[_one(q) for q in state["search_plan"][:8]])
    new_results = "\n---\n".join(parts)
    existing    = state.get("search_results", "")
    return {"search_results": (existing + "\n\n" + new_results).strip()}


def sufficiency_node(state: ResearchState) -> dict:
    """Decide if accumulated search results are enough for a thorough report."""
    result = sufficiency_llm.invoke([
        SystemMessage("Assess whether the research evidence is sufficient for a thorough report."),
        HumanMessage(
            f"Query: {state['query']}\nClarification: {state.get('user_clarification', '')}\n\n"
            f"Search results (truncated):\n{state.get('search_results', '')[:5000]}"
        ),
    ])
    retries = state.get("search_retries", 0)
    return {
        "is_sufficient":  result.is_sufficient,
        "search_retries": retries + (0 if result.is_sufficient else 1),
        "search_plan":    result.additional_queries if not result.is_sufficient else state["search_plan"],
    }


def writer_node(state: ResearchState) -> dict:
    """Write a detailed Markdown research report. Incorporates evaluator feedback on retry."""
    feedback_block = ""
    if state.get("report_feedback"):
        feedback_block = (
            f"\nYour previous draft was rejected. Evaluator feedback:\n{state['report_feedback']}\n"
            "Address every point in this revision."
        )
    response = creative_llm.invoke([
        SystemMessage(
            "You are a senior research writer. Produce a detailed Markdown report "
            "(minimum 1000 words) with: executive summary, background, key findings, "
            "analysis, conclusions, and follow-up questions. Cite sources inline."
        ),
        HumanMessage(
            f"Query: {state['query']}\nClarification: {state.get('user_clarification', '')}\n"
            f"{feedback_block}\n\nResearch evidence:\n{state.get('search_results', '')[:8000]}"
        ),
    ])
    return {"report": response.content}


def evaluator_node(state: ResearchState) -> dict:
    """Score the report 0-10. Approve if score >= 7 and all quality criteria are met."""
    result = evaluator_llm.invoke([
        SystemMessage(
            "Score the research report 0-10. Approve (is_approved=True) only if: "
            "score >= 7, at least 800 words, fully addresses the query, and cites sources."
        ),
        HumanMessage(
            f"Query: {state['query']}\n\nReport:\n{state.get('report', '')}"
        ),
    ])
    retries = state.get("report_retries", 0)
    return {
        "report_score":    result.score,
        "report_feedback": result.feedback,
        "report_approved": result.is_approved,
        "report_retries":  retries + (0 if result.is_approved else 1),
    }


def emailer_node(state: ResearchState) -> dict:
    """Send the approved report via SendGrid."""
    status = send_report_email(
        subject=f"Research Report: {state['query'][:60]}",
        body=state.get("report", ""),
    )
    return {"email_status": status}


# ── Routing functions ──────────────────────────────────────────────────────────

def route_sufficiency(state: ResearchState) -> str:
    """Route to writer if sufficient or retries exhausted; otherwise re-plan."""
    if state.get("is_sufficient") or state.get("search_retries", 0) >= MAX_SEARCH_RETRIES:
        return "writer"
    return "planner"


def route_evaluation(state: ResearchState) -> str:
    """Route to emailer if approved or retries exhausted; otherwise revise."""
    if state.get("report_approved") or state.get("report_retries", 0) >= MAX_REPORT_RETRIES:
        return "emailer"
    return "writer"

In [ ]:
# ── SQLite memory (persistent across sessions and kernel restarts) ──────────────
# SqliteSaver writes checkpoints to a local .db file.
# Each user's thread_id is the key — sessions are fully isolated.
_conn   = sqlite3.connect("research_memory.db", check_same_thread=False)
sql_memory = SqliteSaver(_conn)


# ── Graph 1: Clarifier only ────────────────────────────────────────────────────
clarifier_builder = StateGraph(ResearchState)
clarifier_builder.add_node("clarifier", clarifier_node)
clarifier_builder.add_edge(START, "clarifier")
clarifier_builder.add_edge("clarifier", END)
clarifier_graph = clarifier_builder.compile()


# ── Graph 2: Full research pipeline ───────────────────────────────────────────
research_builder = StateGraph(ResearchState)
research_builder.add_node("planner",     planner_node)
research_builder.add_node("searcher",    searcher_node)
research_builder.add_node("sufficiency", sufficiency_node)
research_builder.add_node("writer",      writer_node)
research_builder.add_node("evaluator",   evaluator_node)
research_builder.add_node("emailer",     emailer_node)

research_builder.add_edge(START,         "planner")
research_builder.add_edge("planner",     "searcher")
research_builder.add_edge("searcher",    "sufficiency")
research_builder.add_conditional_edges("sufficiency", route_sufficiency, {"writer": "writer", "planner": "planner"})
research_builder.add_edge("writer",      "evaluator")
research_builder.add_conditional_edges("evaluator",   route_evaluation,  {"emailer": "emailer", "writer": "writer"})
research_builder.add_edge("emailer",     END)

# SQL checkpointer: each thread_id maps to a unique persistent session
research_graph = research_builder.compile(checkpointer=sql_memory)

display(Image(research_graph.get_graph().draw_mermaid_png()))

In [ ]:
def _initial_state(query: str, user_clarification: str = "") -> ResearchState:
    """Return a fully initialised ResearchState dict."""
    return ResearchState(
        query=query, clarifying_questions=[], context_summary="",
        user_clarification=user_clarification, search_plan=[], search_results="",
        search_retries=0, is_sufficient=False, report="", report_score=0.0,
        report_feedback="", report_retries=0, report_approved=False, email_status="",
    )


async def run_clarifier(query: str) -> tuple[list[str], str]:
    """Phase 1: generate scoping questions. Returns (questions, context_summary)."""
    result = await clarifier_graph.ainvoke(_initial_state(query))
    return result["clarifying_questions"], result["context_summary"]


async def stream_research(query: str, user_clarification: str, thread_id: str):
    """
    Phase 2: stream node transitions and writer output via astream_events.
    Yields (status, report, score_str, email_str) tuples for Gradio.
    Uses the SQLite checkpointer so each thread_id is a persistent session.
    """
    config = {"configurable": {"thread_id": thread_id}}
    state  = _initial_state(query, user_clarification)
    status = "Safety check passed. Starting pipeline...\n"
    report = ""

    try:
        # astream_events version="v2" emits per-node and per-token events
        async for event in research_graph.astream_events(state, config, version="v2"):
            kind = event["event"]
            node = event.get("metadata", {}).get("langgraph_node", "")

            if kind == "on_chain_start" and node in _PIPELINE_NODES:
                status += f"Running: {node}...\n"
                yield status, report, "", ""

            elif kind == "on_chat_model_stream" and node == "writer":
                # Stream the writer's tokens directly to the report panel
                chunk = event["data"]["chunk"]
                if hasattr(chunk, "content") and chunk.content:
                    report += chunk.content
                    yield status, report, "", ""

        # Retrieve final state values from the SQLite checkpoint
        final        = research_graph.get_state(config)
        score        = final.values.get("report_score", 0.0)
        email_status = final.values.get("email_status", "")

        # Output guardrail: heuristic quality check
        passed, msg = check_report_quality(report)
        if not passed:
            status += f"Output guardrail warning: {msg}\n"

        wc = len(report.split())
        status += f"Done. {wc} words | Score: {score}/10 | Email: {email_status}\n"
        yield status, report, f"{score}/10", email_status

    except Exception as exc:
        yield f"Pipeline error: {exc}\n{status}", report, "", ""


async def load_session(thread_id: str) -> tuple[str, str]:
    """Load a previous session from SQLite by thread_id. Returns (report, score_str)."""
    try:
        config   = {"configurable": {"thread_id": thread_id.strip()}}
        snapshot = research_graph.get_state(config)
        if not snapshot or not snapshot.values:
            return "Session not found — check the ID.", ""
        report = snapshot.values.get("report", "No report in this session.")
        score  = snapshot.values.get("report_score", 0.0)
        return report, f"{score}/10"
    except Exception as exc:
        return f"Error loading session: {exc}", ""

In [ ]:
# ── Gradio UI ──────────────────────────────────────────────────────────────────

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald"), title="Deep Research Agent") as demo:
    gr.Markdown(
        "# Deep Research Agent\n"
        "Enter a topic. The agent clarifies scope, researches in parallel, writes and evaluates "
        "a report, then emails it.  Each browser session is isolated — sessions persist across restarts."
    )

    # gr.State holds per-user data — different browser tabs get different values
    thread_state    = gr.State(lambda: str(uuid.uuid4()))
    questions_state = gr.State([])

    # Session management row
    with gr.Row():
        session_label  = gr.Textbox(label="Your session ID (copy to resume later)", interactive=False, scale=3)
        resume_box     = gr.Textbox(label="Resume a previous session ID", placeholder="Paste a session ID here", scale=3)
        resume_btn     = gr.Button("Load session", scale=1)

    # Research input
    with gr.Row():
        query_box   = gr.Textbox(label="Research topic",
                                 placeholder="e.g. What are the latest advances in fusion energy?", lines=2)
        clarify_btn = gr.Button("Get Scope Questions", variant="secondary")

    # Clarification phase (revealed after step 1)
    with gr.Group(visible=False) as clarify_group:
        gr.Markdown("### Scope — select or edit one of the questions, then add context")
        questions_display = gr.Markdown()
        clarification_box = gr.Textbox(
            label="Your clarification (edit or rephrase a question above)",
            placeholder="e.g. Focus on tokamak advances in the past 2 years", lines=2,
        )
        extra_context_box = gr.Textbox(
            label="Additional context (optional)",
            placeholder="e.g. Include commercial projects and recent government funding", lines=2,
        )
        research_btn = gr.Button("Start Research", variant="primary", size="lg")

    # Pipeline status
    with gr.Row():
        status_box = gr.Textbox(label="Pipeline status", lines=8, interactive=False,
                                placeholder="Status updates will appear here...")

    # Results
    with gr.Row():
        with gr.Column(scale=3):
            report_box = gr.Markdown("The report will stream here as the writer works...")
        with gr.Column(scale=1):
            score_box = gr.Textbox(label="Evaluator score", interactive=False)
            email_box = gr.Textbox(label="Email status",    interactive=False)

    # ── Event handlers ─────────────────────────────────────────────────────────

    # Show session ID when UI loads
    def show_session(thread):
        return thread
    demo.load(show_session, inputs=[thread_state], outputs=[session_label])

    # Resume previous session
    async def do_resume(resume_id):
        if not resume_id.strip():
            return "", "Please paste a session ID.", "", ""
        report, score = await load_session(resume_id)
        return resume_id.strip(), report, score, ""

    resume_btn.click(
        do_resume,
        inputs=[resume_box],
        outputs=[session_label, report_box, score_box, email_box],
    )

    # Phase 1: run clarifier, show questions
    async def do_clarify(query, thread):
        if not query.strip():
            return gr.update(), "", "", [], thread
        # Input guardrail
        safety = await asyncio.to_thread(check_query_safety, query.strip())
        if not safety.is_safe:
            return (
                gr.update(visible=True),
                f"Query blocked by safety check: {safety.reason}",
                "", [], thread,
            )
        try:
            questions, summary = await run_clarifier(query.strip())
            q_md    = f"**{summary}**\n\n" + "\n\n".join(f"{i+1}. {q}" for i, q in enumerate(questions))
            prefill = questions[0] if questions else ""
            new_thread = str(uuid.uuid4())  # fresh thread_id for each new query
            return gr.update(visible=True), q_md, prefill, questions, new_thread
        except Exception as exc:
            return gr.update(visible=True), f"Clarifier error: {exc}", "", [], thread

    _clarify_outputs = [clarify_group, questions_display, clarification_box, questions_state, thread_state]
    clarify_btn.click(do_clarify, inputs=[query_box, thread_state], outputs=_clarify_outputs)
    query_box.submit(do_clarify,  inputs=[query_box, thread_state], outputs=_clarify_outputs)

    # Phase 2: stream the full pipeline
    async def do_research(query, clarification, extra_context, thread):
        if not query.strip():
            yield "Please enter a research topic.", "", "", ""
            return
        full_clarification = clarification.strip()
        if extra_context.strip():
            full_clarification += f"\n\nAdditional context: {extra_context.strip()}"
        async for status, report, score, email in stream_research(query, full_clarification, thread):
            yield status, report, score, email

    research_btn.click(
        do_research,
        inputs=[query_box, clarification_box, extra_context_box, thread_state],
        outputs=[status_box, report_box, score_box, email_box],
    )

demo.launch(inbrowser=True)